# 07 Exploratory Data Analysis

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

# Notebook 07 — Exploratory Data Analysis (EDA)

### Indian Cinema Top-Grossing Movies Dataset (2000–2025)

This notebook performs exploratory data analysis on the finalized and processed movie dataset.

Objectives:
- Understand dataset structure and quality
- Analyze yearly and language-wise movie distribution
- Examine India Gross and Worldwide Gross
- Identify highest-grossing movies and trends
- Analyze genre and certification patterns
- Examine relationships between movie attributes and box-office performance
- Generate insights for subsequent stages of the project

In [ ]:
# ============================================================
# RESET LOCAL DRIVE MOUNTPOINT
# ============================================================

import os
import shutil

if os.path.exists("/content/drive"):
    print("Current /content/drive contents:")
    print(os.listdir("/content/drive"))

    shutil.rmtree("/content/drive")

    print("\nStale local mountpoint removed.")
else:
    print("/content/drive does not exist.")

print(
    "Mountpoint exists after cleanup:",
    os.path.exists("/content/drive")
)

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ============================================================
# PROJECT ROOT
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

print("Project Root:", PROJECT_ROOT)
print("Project exists:", os.path.exists(PROJECT_ROOT))

In [ ]:
# ============================================================
# PROJECT STRUCTURE
# ============================================================

print("=" * 70)
print("PROJECT ROOT CONTENTS")
print("=" * 70)

for item in os.listdir(PROJECT_ROOT):
    full_path = os.path.join(PROJECT_ROOT, item)

    if os.path.isdir(full_path):
        print(f"[DIR]  {item}")
    else:
        print(f"[FILE] {item}")

In [ ]:
# ============================================================
# DATA PATHS
# ============================================================

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data"
)

PROCESSED_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

print("Data Directory:")
print(DATA_DIR)

print("\nProcessed Directory:")
print(PROCESSED_DIR)

print("\nPath verification:")
print("Data Directory exists:", os.path.exists(DATA_DIR))
print("Processed Directory exists:", os.path.exists(PROCESSED_DIR))

In [ ]:
# ============================================================
# INSPECT PROCESSED DATA
# ============================================================

print("=" * 70)
print("PROCESSED DATA CONTENTS")
print("=" * 70)

if os.path.exists(PROCESSED_DIR):

    for item in os.listdir(PROCESSED_DIR):
        full_path = os.path.join(PROCESSED_DIR, item)

        if os.path.isdir(full_path):
            print(f"[DIR]  {item}")
        else:
            print(f"[FILE] {item}")

else:
    print("Processed directory does not exist.")

In [ ]:
# ============================================================
# LOAD FINAL FEATURE DATASET
# ============================================================

FEATURES_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_FEATURES.csv"
)

df = pd.read_csv(FEATURES_PATH)

print("Feature dataset loaded successfully.")
print("Dataset path:", FEATURES_PATH)
print("Dataset shape:", df.shape)

In [ ]:
# ============================================================
# COLUMN NAMES
# ============================================================

print("=" * 70)
print("DATASET COLUMNS")
print("=" * 70)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

In [ ]:
# ============================================================
# FIRST 10 RECORDS
# ============================================================

df.head(10)

In [ ]:
# ============================================================
# DATASET INFORMATION
# ============================================================

df.info()

In [ ]:
# ============================================================
# BASIC DATASET STATISTICS
# ============================================================

print("=" * 70)
print("BASIC DATASET STATISTICS")
print("=" * 70)

print(f"Total movies       : {len(df)}")
print(f"Total features     : {df.shape[1]}")
print(f"Years covered      : {df['year'].min()} - {df['year'].max()}")
print(f"Number of years    : {df['year'].nunique()}")
print(f"Languages          : {df['language'].nunique()}")
print(f"Unique movie titles: {df['movie_title'].nunique()}")

In [ ]:
# ============================================================
# MISSING VALUE ANALYSIS
# ============================================================

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary
    .sort_values("missing_count", ascending=False)
)

missing_summary

In [ ]:
# ============================================================
# TMDB FEATURE AVAILABILITY
# ============================================================

availability_summary = pd.DataFrame({
    "Feature": [
        "Budget",
        "Worldwide Revenue"
    ],
    "Available_Count": [
        df["budget_available"].sum(),
        df["worldwide_revenue_available"].sum()
    ],
    "Unavailable_Count": [
        (~df["budget_available"]).sum(),
        (~df["worldwide_revenue_available"]).sum()
    ]
})

availability_summary["Available_Percentage"] = (
    availability_summary["Available_Count"]
    / len(df) * 100
).round(2)

availability_summary["Unavailable_Percentage"] = (
    availability_summary["Unavailable_Count"]
    / len(df) * 100
).round(2)

availability_summary

In [ ]:
# ============================================================
# VERIFY AVAILABILITY FLAGS
# ============================================================

budget_flag_check = (
    df["budget_available"]
    == (df["tmdb_budget_usd"] > 0)
)

revenue_flag_check = (
    df["worldwide_revenue_available"]
    == (df["tmdb_worldwide_revenue_usd"] > 0)
)

print(
    "Budget availability flag mismatches:",
    (~budget_flag_check).sum()
)

print(
    "Worldwide revenue availability flag mismatches:",
    (~revenue_flag_check).sum()
)

In [ ]:
# ============================================================
# DUPLICATE ANALYSIS
# ============================================================

print("Duplicate complete rows:", df.duplicated().sum())

print(
    "Duplicate movie_record_id:",
    df["movie_record_id"].duplicated().sum()
)

print(
    "Duplicate movie titles:",
    df["movie_title"].duplicated().sum()
)

In [ ]:
# ============================================================
# MOVIES PER YEAR
# ============================================================

movies_per_year = (
    df.groupby("year")
      .size()
      .reset_index(name="movie_count")
)

movies_per_year

In [ ]:
# ============================================================
# VERIFY TOP 5 PER YEAR
# ============================================================

print(
    "Years with exactly 5 movies:",
    (movies_per_year["movie_count"] == 5).sum()
)

print(
    "Years with fewer than 5 movies:",
    (movies_per_year["movie_count"] < 5).sum()
)

print(
    "Years with more than 5 movies:",
    (movies_per_year["movie_count"] > 5).sum()
)

print("\nYears that do not contain exactly 5 movies:")

movies_per_year[
    movies_per_year["movie_count"] != 5
]

In [ ]:
# ============================================================
# VERIFY YEARLY RANK
# ============================================================

rank_check = (
    df.groupby("year")["rank"]
      .apply(lambda x: sorted(x.tolist()))
)

rank_check

In [ ]:
# ============================================================
# VERIFY MOVIE RECORD IDs
# ============================================================

expected_ids = (
    df["year"].astype(str)
    + "_"
    + df["rank"].astype(str)
)

print(
    "Movie record ID mismatches:",
    (df["movie_record_id"] != expected_ids).sum()
)

In [ ]:
# ============================================================
# MOVIES PER YEAR
# ============================================================

plt.figure(figsize=(14, 6))

plt.bar(
    movies_per_year["year"],
    movies_per_year["movie_count"]
)

plt.title("Number of Movies per Year")
plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.xticks(movies_per_year["year"], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# LANGUAGE DISTRIBUTION
# ============================================================

language_counts = (
    df["language"]
    .value_counts()
)

language_counts

In [ ]:
# ============================================================
# LANGUAGE DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    language_counts.index,
    language_counts.values
)

plt.title("Language Distribution in ICMB-130")
plt.xlabel("Language")
plt.ylabel("Number of Movies")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# LANGUAGE DISTRIBUTION BY YEAR
# ============================================================

language_by_year = pd.crosstab(
    df["year"],
    df["language"]
)

language_by_year

In [ ]:
# ============================================================
# LANGUAGE SHARE
# ============================================================

language_share = (
    df["language"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)

language_share.columns = [
    "language",
    "percentage"
]

language_share

# India Gross Descriptive Statistics

In [ ]:
# ============================================================
# INDIA GROSS DESCRIPTIVE STATISTICS
# ============================================================

india_gross_stats = (
    df["india_gross_inr_cr"]
    .describe()
)

india_gross_stats

In [ ]:
# ============================================================
# INDIA GROSS BY YEAR
# ============================================================

india_gross_by_year = (
    df.groupby("year")["india_gross_inr_cr"]
      .agg(
          total="sum",
          mean="mean",
          median="median",
          minimum="min",
          maximum="max"
      )
      .reset_index()
)

india_gross_by_year

In [ ]:
# ============================================================
# INDIA GROSS TREND
# ============================================================

plt.figure(figsize=(14, 6))

plt.plot(
    india_gross_by_year["year"],
    india_gross_by_year["mean"],
    marker="o",
    label="Mean India Gross"
)

plt.plot(
    india_gross_by_year["year"],
    india_gross_by_year["median"],
    marker="o",
    label="Median India Gross"
)

plt.title("India Gross Trend of Top 5 Movies by Year")
plt.xlabel("Year")
plt.ylabel("India Gross (₹ Crore)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TOP 15 MOVIES BY INDIA GROSS
# ============================================================

top_india_gross = (
    df.sort_values(
        "india_gross_inr_cr",
        ascending=False
    )
    .head(15)
)

top_india_gross[
    [
        "year",
        "rank",
        "movie_title",
        "language",
        "india_gross_inr_cr"
    ]
]

In [ ]:
# ============================================================
# INDIA GROSS DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["india_gross_inr_cr"],
    bins=20
)

plt.title("Distribution of India Gross")
plt.xlabel("India Gross (₹ Crore)")
plt.ylabel("Number of Movies")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# LOG INDIA GROSS DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["log_india_gross"],
    bins=20
)

plt.title("Distribution of Log-Transformed India Gross")
plt.xlabel("Log India Gross")
plt.ylabel("Number of Movies")

plt.tight_layout()
plt.show()

## Rank-wise India Gross Analysis

how India Gross varies by yearly rank across the Top 5 movies.

In [ ]:
# Rank-wise India Gross statistics

rank_gross = (
    df.groupby("rank")["india_gross_inr_cr"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

display(rank_gross)

In [ ]:
plt.figure(figsize=(10, 5))

rank_gross_plot = (
    df.groupby("rank")["india_gross_inr_cr"]
      .mean()
)

plt.plot(
    rank_gross_plot.index,
    rank_gross_plot.values,
    marker="o"
)

plt.title("Average India Gross by Yearly Rank")
plt.xlabel("Rank")
plt.ylabel("Average India Gross (₹ Crore)")
plt.xticks(range(1, 6))
plt.grid(True, alpha=0.3)

plt.show()

## Language-wise India Gross Analysis

Compare box-office performance across the languages represented in ICMB-130.

In [ ]:
language_gross = (
    df.groupby("language")["india_gross_inr_cr"]
      .agg(["count", "mean", "median", "min", "max", "sum"])
      .sort_values("mean", ascending=False)
      .round(2)
)

display(language_gross)

In [ ]:
plt.figure(figsize=(9, 5))

language_mean_gross = (
    df.groupby("language")["india_gross_inr_cr"]
      .mean()
      .sort_values(ascending=False)
)

plt.bar(
    language_mean_gross.index,
    language_mean_gross.values
)

plt.title("Average India Gross by Language")
plt.xlabel("Language")
plt.ylabel("Average India Gross (₹ Crore)")
plt.xticks(rotation=30)

plt.show()

## Budget Distribution

Analyze the distribution of TMDB-reported movie budgets among films where budget data is available.

In [ ]:
budget_data = df.loc[
    df["budget_available"],
    "tmdb_budget_inr_cr"
]

print("Movies with available budget:", len(budget_data))
print()

display(
    budget_data.describe()
    .round(2)
    .to_frame(name="Budget (₹ Crore)")
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(budget_data, bins=15)

plt.title("Distribution of Movie Budget")
plt.xlabel("Budget (₹ Crore)")
plt.ylabel("Number of Movies")

plt.show()

## Distribution of Log-Transformed Budget

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df["log_budget"].dropna(),
    bins=15
)

plt.title("Distribution of Log-Transformed Budget")
plt.xlabel("Log Budget")
plt.ylabel("Number of Movies")

plt.show()

## Worldwide Revenue Distribution

Analyze worldwide revenue for movies where TMDB revenue data is available.

In [ ]:
worldwide_data = df.loc[
    df["worldwide_revenue_available"],
    "tmdb_worldwide_revenue_inr_cr"
]

print("Movies with available worldwide revenue:", len(worldwide_data))
print()

display(
    worldwide_data.describe()
    .round(2)
    .to_frame(name="Worldwide Revenue (₹ Crore)")
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(worldwide_data, bins=15)

plt.title("Distribution of Worldwide Revenue")
plt.xlabel("Worldwide Revenue (₹ Crore)")
plt.ylabel("Number of Movies")

plt.show()

## Revenue-to-Budget Ratio Analysis

Examine the worldwide revenue generated relative to the reported production budget.

In [ ]:
ratio_data = df["revenue_budget_ratio"].dropna()

print("Movies with valid revenue-budget ratio:", len(ratio_data))
print()

display(
    ratio_data.describe()
    .round(2)
    .to_frame(name="Revenue/Budget Ratio")
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(ratio_data, bins=15)

plt.title("Distribution of Worldwide Revenue-to-Budget Ratio")
plt.xlabel("Worldwide Revenue / Budget")
plt.ylabel("Number of Movies")

plt.show()

## India Gross Share of Worldwide Revenue

Analyze the proportion of worldwide revenue represented by India Gross.

In [ ]:
share_data = df["india_gross_share_pct"].dropna()

print("Movies with valid India Gross share:", len(share_data))
print()

display(
    share_data.describe()
    .round(2)
    .to_frame(name="India Gross Share (%)")
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(share_data, bins=15)

plt.title("Distribution of India Gross Share of Worldwide Revenue")
plt.xlabel("India Gross Share (%)")
plt.ylabel("Number of Movies")

plt.show()

## India Gross vs Worldwide Revenue

Examine the relationship between domestic India Gross and worldwide revenue.

In [ ]:
plot_data = df[
    ["india_gross_inr_cr", "tmdb_worldwide_revenue_inr_cr"]
].dropna()

plt.figure(figsize=(9, 6))

plt.scatter(
    plot_data["india_gross_inr_cr"],
    plot_data["tmdb_worldwide_revenue_inr_cr"]
)

plt.title("India Gross vs Worldwide Revenue")
plt.xlabel("India Gross (₹ Crore)")
plt.ylabel("Worldwide Revenue (₹ Crore)")

plt.show()

## Log India Gross vs Log Worldwide Revenue

In [ ]:
plot_data = df[
    ["log_india_gross", "log_worldwide_revenue"]
].dropna()

plt.figure(figsize=(9, 6))

plt.scatter(
    plot_data["log_india_gross"],
    plot_data["log_worldwide_revenue"]
)

plt.title("Log India Gross vs Log Worldwide Revenue")
plt.xlabel("Log India Gross")
plt.ylabel("Log Worldwide Revenue")

plt.show()

## Correlation Analysis

Examine relationships among the major numerical financial features.

In [ ]:
correlation_columns = [
    "year",
    "rank",
    "india_gross_inr_cr",
    "tmdb_budget_inr_cr",
    "tmdb_worldwide_revenue_inr_cr",
    "revenue_budget_ratio",
    "india_gross_share_pct",
    "log_india_gross",
    "log_worldwide_revenue",
    "log_budget"
]

correlation_matrix = (
    df[correlation_columns]
    .corr()
    .round(2)
)

display(correlation_matrix)

In [ ]:
plt.figure(figsize=(12, 9))

plt.imshow(
    correlation_matrix,
    aspect="auto"
)

plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=75,
    ha="right"
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

plt.title("Correlation Matrix of Major Numerical Features")

plt.tight_layout()
plt.show()

## India Gross Outlier Detection

Identify unusually high India Gross values using the IQR method.

In [ ]:
Q1 = df["india_gross_inr_cr"].quantile(0.25)
Q3 = df["india_gross_inr_cr"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

india_gross_outliers = df[
    (df["india_gross_inr_cr"] < lower_bound) |
    (df["india_gross_inr_cr"] > upper_bound)
].copy()

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower Bound: {lower_bound:.2f}")
print(f"Upper Bound: {upper_bound:.2f}")
print(f"\nNumber of outliers: {len(india_gross_outliers)}")

display(
    india_gross_outliers[
        [
            "year",
            "rank",
            "movie_title",
            "language",
            "india_gross_inr_cr"
        ]
    ]
    .sort_values("india_gross_inr_cr", ascending=False)
)

## Year-over-Year Growth in Average India Gross

Measure how the average India Gross of the yearly Top 5 changed over time.

In [ ]:
yearly_mean = (
    df.groupby("year")["india_gross_inr_cr"]
      .mean()
      .sort_index()
)

yoy_growth = yearly_mean.pct_change() * 100

yoy_analysis = pd.DataFrame({
    "year": yearly_mean.index,
    "average_india_gross": yearly_mean.values,
    "yoy_growth_pct": yoy_growth.values
})

yoy_analysis["average_india_gross"] = (
    yoy_analysis["average_india_gross"].round(2)
)

yoy_analysis["yoy_growth_pct"] = (
    yoy_analysis["yoy_growth_pct"].round(2)
)

display(yoy_analysis)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    yoy_analysis["year"],
    yoy_analysis["yoy_growth_pct"],
    marker="o"
)

plt.axhline(0, linewidth=1)

plt.title("Year-over-Year Growth in Average India Gross")
plt.xlabel("Year")
plt.ylabel("YoY Growth (%)")

plt.grid(True, alpha=0.3)

plt.show()

## Final EDA Summary

Summary of the key structural, financial, and temporal characteristics of the ICMB-130 dataset.

In [ ]:
# ============================================================
# FINAL EDA SUMMARY
# ============================================================

summary = {
    "Total Movies": len(df),
    "Year Range": f"{df['year'].min()}–{df['year'].max()}",
    "Number of Years": df["year"].nunique(),
    "Movies per Year": df.groupby("year").size().unique().tolist(),
    "Languages": df["language"].nunique(),
    "Missing Values": int(df.isna().sum().sum()),
    "Duplicate Movie IDs": int(df["movie_record_id"].duplicated().sum()),
    "Budget Available": int(df["budget_available"].sum()),
    "Worldwide Revenue Available": int(df["worldwide_revenue_available"].sum()),
    "Mean India Gross (₹ Cr)": round(df["india_gross_inr_cr"].mean(), 2),
    "Median India Gross (₹ Cr)": round(df["india_gross_inr_cr"].median(), 2),
    "Maximum India Gross (₹ Cr)": round(df["india_gross_inr_cr"].max(), 2),
}

summary_df = pd.DataFrame(
    summary.items(),
    columns=["Metric", "Value"]
)

display(summary_df)

## Feature Readiness Summary

Review the availability of the major financial features before downstream analysis.

In [ ]:
# Major financial feature availability

feature_readiness = pd.DataFrame({
    "Feature": [
        "India Gross",
        "Budget",
        "Worldwide Revenue",
        "Revenue/Budget Ratio",
        "India Gross Share",
        "Log India Gross",
        "Log Worldwide Revenue",
        "Log Budget"
    ],
    "Available": [
        df["india_gross_inr_cr"].notna().sum(),
        df["tmdb_budget_inr_cr"].notna().sum(),
        df["tmdb_worldwide_revenue_inr_cr"].notna().sum(),
        df["revenue_budget_ratio"].notna().sum(),
        df["india_gross_share_pct"].notna().sum(),
        df["log_india_gross"].notna().sum(),
        df["log_worldwide_revenue"].notna().sum(),
        df["log_budget"].notna().sum()
    ],
    "Missing": [
        df["india_gross_inr_cr"].isna().sum(),
        df["tmdb_budget_inr_cr"].isna().sum(),
        df["tmdb_worldwide_revenue_inr_cr"].isna().sum(),
        df["revenue_budget_ratio"].isna().sum(),
        df["india_gross_share_pct"].isna().sum(),
        df["log_india_gross"].isna().sum(),
        df["log_worldwide_revenue"].isna().sum(),
        df["log_budget"].isna().sum()
    ]
})

feature_readiness["Available_%"] = (
    feature_readiness["Available"] / len(df) * 100
).round(2)

display(feature_readiness)

## Final Dataset Integrity Check

Confirm that the dataset remains structurally valid after EDA.

In [ ]:
# Final integrity checks

assert len(df) == 130
assert df["movie_record_id"].is_unique
assert df.groupby("year").size().eq(5).all()
assert df["rank"].between(1, 5).all()

print("============================================================")
print("FINAL EDA DATASET INTEGRITY CHECK")
print("============================================================")
print("Total records:", len(df))
print("Unique movie IDs:", df["movie_record_id"].nunique())
print("Years:", df["year"].min(), "to", df["year"].max())
print("Exactly 5 movies per year:", df.groupby("year").size().eq(5).all())
print("Ranks valid (1–5):", df["rank"].between(1, 5).all())
print("Dataset integrity check: PASSED")

In [ ]:
from pathlib import Path
import zipfile
import re
import pandas as pd

PROJECT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
ARCHIVE_PATH = PROJECT_DIR / "01_data/indicdialogue/dataset.zip"

FILES_TO_INSPECT = [
    "IndicDialogue/Hindi/subtitles/1955417232.srt",
    "IndicDialogue/Hindi/subtitles/1955654515.srt",
    "IndicDialogue/Hindi/subtitles/1956166888.srt",
    "IndicDialogue/Hindi/subtitles/1956297871.srt",
    "IndicDialogue/Hindi/subtitles/1957595445.srt",
    "IndicDialogue/Tamil/subtitles/1955328184.srt",
    "IndicDialogue/Tamil/subtitles/1958231917.srt",
]

def decode_subtitle(raw_bytes):
    """Try common encodings without altering the source bytes."""
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return raw_bytes.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    return raw_bytes.decode("utf-8", errors="replace"), "utf-8-replacement"

def inspect_subtitle(text):
    lines = text.splitlines()

    timestamp_pattern = re.compile(
        r"\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*"
        r"\d{2}:\d{2}:\d{2}[,.]\d{3}"
    )

    timestamps = [
        line.strip()
        for line in lines
        if timestamp_pattern.search(line)
    ]

    dialogue_lines = []
    for line in lines:
        stripped = line.strip()
        if (
            not stripped
            or timestamp_pattern.search(stripped)
            or stripped.isdigit()
            or stripped.startswith(("WEBVTT", "NOTE "))
        ):
            continue
        dialogue_lines.append(stripped)

    dialogue_text = " ".join(dialogue_lines)
    words = re.findall(r"\b[\w'-]+\b", dialogue_text, flags=re.UNICODE)

    # Count Unicode-script characters as evidence, not as definitive
    # language identification.
    devanagari = len(re.findall(r"[\u0900-\u097F]", dialogue_text))
    tamil = len(re.findall(r"[\u0B80-\u0BFF]", dialogue_text))
    sinhala = len(re.findall(r"[\u0D80-\u0DFF]", dialogue_text))
    latin = len(re.findall(r"[A-Za-z]", dialogue_text))

    return {
        "dialogue_blocks": len(timestamps),
        "dialogue_lines": len(dialogue_lines),
        "word_count": len(words),
        "character_count": len(dialogue_text),
        "timestamp_start": timestamps[0].split("-->")[0].strip()
            if timestamps else None,
        "timestamp_end": timestamps[-1].split("-->")[-1].strip()
            if timestamps else None,
        "devanagari_chars": devanagari,
        "tamil_chars": tamil,
        "sinhala_chars": sinhala,
        "latin_chars": latin,
        "first_dialogue_samples": dialogue_lines[:5],
        "last_dialogue_samples": dialogue_lines[-3:],
    }

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    available = set(zf.namelist())
    results = []

    for member in FILES_TO_INSPECT:
        print("\n" + "=" * 85)
        print(member)

        if member not in available:
            print("MISSING FROM ARCHIVE")
            continue

        raw_bytes = zf.read(member)
        text, encoding = decode_subtitle(raw_bytes)
        details = inspect_subtitle(text)

        print("File size:", f"{len(raw_bytes):,}", "bytes")
        print("Encoding used:", encoding)
        print("Dialogue blocks:", details["dialogue_blocks"])
        print("Word count:", details["word_count"])
        print("Timestamp range:",
              details["timestamp_start"], "to", details["timestamp_end"])
        print(
            "Script counts — Devanagari:",
            details["devanagari_chars"],
            "| Tamil:", details["tamil_chars"],
            "| Sinhala:", details["sinhala_chars"],
            "| Latin:", details["latin_chars"],
        )
        print("First dialogue samples:")
        for sample in details["first_dialogue_samples"]:
            print("  ", sample[:250])
        print("Last dialogue samples:")
        for sample in details["last_dialogue_samples"]:
            print("  ", sample[:250])

        results.append({
            "archive_path": member,
            "file_size_bytes": len(raw_bytes),
            "encoding": encoding,
            **details,
        })

print("\n" + "=" * 85)
print("Inspection complete. No files extracted or modified.")